# Python for Bioinformatics — Part II Exercises
## Modules 6–8 · Real Programming

Work alongside the theory document, in order, following the green **▶ Practise now** pointers.

**Setup:** make sure the `data/` folder (with `sample.fasta`, `sample.fastq`, `variants.csv`) sits in the **same folder** as this notebook. Run the cell below to confirm.

In [18]:
import os
for fn in ["data/sample.fasta", "data/sample.fastq", "data/variants.csv"]:
    print(fn, "found" if os.path.exists(fn) else "MISSING — check folder location")

data/sample.fasta found
data/sample.fastq found
data/variants.csv found


---
# Module 6 — Reading & writing files

## 6A. Reading basics
### Worked example

In [24]:
with open("data/sample.fasta") as f:
    text = f.read()
print("characters:", len(text))
print("first 50:", text[:50])

characters: 495
first 50: >rpoB Salmonella Paratyphi B RNA polymerase beta s


### 6A.1 (easy)
Open `data/sample.fasta` and, iterating line by line, count how many lines it has and how many are **header** lines (start with `>`). Print both counts. Remember to `.rstrip("\n")`.

In [30]:
# your code here
count = 0
header = 0
with open("data/sample.fasta") as f:
    for line in f:
        line = line.rstrip("\n")
        count += 1
        if line[0] == ">":
            header += 1

print(count, header)

10 3


### 6A.2 (harder)
Read `data/sample.fasta` with `.readlines()`. Print the number of lines, then print only the header lines (those starting with `>`), with the leading `>` removed.

In [32]:
# your code here
count = 0
with open("data/sample.fasta") as f:
    for l in f:
        line = l.rstrip("\n")
        count += 1
        if line[0] == ">":
            print(line[1:])
print(f"Number of lines: {count}")

rpoB Salmonella Paratyphi B RNA polymerase beta subunit
katG catalase-peroxidase
gyrA DNA gyrase subunit A
Number of lines: 10


## 6B. Parsing FASTA (core exercise)
### Worked example — the canonical parser

In [1]:
def read_fasta(path):
    """Parse FASTA into {header: sequence}."""
    sequences, header, parts = {}, None, []
    with open(path) as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if header is not None:
                    sequences[header] = "".join(parts)
                header = line[1:]
                parts = []
            else:
                parts.append(line)
        if header is not None:
            sequences[header] = "".join(parts)
    return sequences

records = read_fasta("data/sample.fasta")
for h, s in records.items():
    print(h[:20], "...", len(s), "bp")

rpoB Salmonella Para ... 120 bp
katG catalase-peroxi ... 150 bp
gyrA DNA gyrase subu ... 108 bp


### 6B.1 (easy)
Using the `read_fasta` function above, print: how many sequences the file contains, and the length of the **katG** sequence (its header starts with `katG`). Hint: loop over `records.items()` and check `header.startswith("katG")`.

In [9]:
records = read_fasta("data/sample.fasta")
# your code here
print(records)
print("Number of sequences:",len(records))
katG_length = len([header for header in records if header.startswith("katG")][0])
print(f"Length of katG: {katG_length}")

{'rpoB Salmonella Paratyphi B RNA polymerase beta subunit': 'ATGGTTTACTCCTATACCGAGAAAAAACGTATTCGTAAGGATTTTGGTAAACGTCCGCAAGTTCTGGATGTACCGTATCTGCTGTCCATCCAGCTGGACTCCTTCCAGAAATTCCTGGAT', 'katG catalase-peroxidase': 'ATGCCCGAGCAGCACCCGCCGATTACCGAGACCACCACCGGCGCCGCCTCCAACGGCTGCGACCCGGTGGCCGCCGGCGAGCGCTGGTGGCCGAACCGCCTGAACCTGAAGGTGCTGCACCAGAACCCGGCGGTGGCCGACCCGATGGGC', 'gyrA DNA gyrase subunit A': 'ATGAGCGACCTTGCGAGAGAAATTACACCGGTCAACATTGAGGAAGAGCTGAAGAGCTCCTATCTGGATTATGCGATGTCGGTCATTGTTGGCCGTGCGCTGCCAGAT'}
Number of sequences: 3
Length of katG: 24


### 6B.2 (harder)
For every record, compute GC content and print a line `gene_id: NN.N% GC`. Use only the gene name (the first word of the header) — `header.split()[0]` gives it.

In [10]:
records = read_fasta("data/sample.fasta")
# your code here

for gen, seq in records.items():
    gc = (seq.count("G") + seq.count("C")) / len(seq)
    print(f"{gen.split()[0]} | {gc:.1%} GC")

rpoB | 44.2% GC
katG | 73.3% GC
gyrA | 50.9% GC


## 6C. Parsing FASTQ
### Worked example — read four lines at a time

In [14]:
def read_fastq(path):
    """Yield (header, seq, qual) tuples from a FASTQ file."""
    reads = []
    with open(path) as f:
        lines = [ln.rstrip("\n") for ln in f]
    for i in range(0, len(lines), 4):
        header = lines[i][1:]      # drop '@'
        seq    = lines[i+1]
        qual   = lines[i+3]        # lines[i+2] is the '+'
        reads.append((header, seq, qual))
    return reads

for header, seq, qual in read_fastq("data/sample.fastq"):
    print(header, seq, qual)

read1 ATGCGTACGTTAGC IIIIIIIIIIIIII
read2 GGCATTAGCCGTAA IIIIIHHHGGFFEE
read3 TTTAAACCCGGGAT ##$%&'()*+,-./
read4 ACGTACGTACGTAC IIIIIIIIIIIIII


### 6C.1 (easy)
Write `phred(char)` returning the Phred+33 score (`ord(char) - 33`). Test it: print the score of `"I"` (should be 40) and `"#"` (should be 2).

In [18]:
# your code here
def phred(char):
    return (ord(char) - 33)

print(phred("I"),",", phred("#"))

40 , 2


### 6C.2 (harder)
For each read in `data/sample.fastq`, compute its **mean quality** (average Phred+33 score across the quality string) and print `read_id: mean_qual` rounded to one decimal.

In [32]:
reads = read_fastq("data/sample.fastq")
# your code here
def mean_quality(q):
    mean = sum(phred(p) for p in q)/len(q)
    return mean

for read, seq, q in reads:
    print(f"{read.split()[0]}: {mean_quality(q):.1f}")


read1: 40.0
read2: 38.5
read3: 7.6
read4: 40.0


### 6C.3 (challenge — quality filter)
Print only the reads whose mean quality is **>= 30**, showing the read id and its mean quality. (read3 should be filtered out.)

In [33]:
reads = read_fastq("data/sample.fastq")
# your code here

for read, seq, q in reads:
    mean = mean_quality(q)
    if mean >= 30:
        print(f"{read.split()[0]}: {mean:.1f}")

read1: 40.0
read2: 38.5
read4: 40.0


## 6D. CSV / tabular data
### Worked example

In [38]:
import csv
with open("data/variants.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row["gene"], row["position"], row["quality"])

rpoB 1349 45.2
katG 944 38.7
gyrA 248 52.1
gyrA 259 29.4
pncA 169 41.0


### 6D.1 (easy)
Read `data/variants.csv` with `csv.DictReader` and print only the `gene` and `alt` allele of each variant.

In [ ]:
# your code here
with open("data/variants.csv") as f:
    reader = csv.DictReader(f)
    for l in reader:
        print(f"{l["gene"]} | {l["alt"]}")

{'gene': 'rpoB', 'position': '1349', 'ref': 'C', 'alt': 'T', 'quality': '45.2'}
1349
rpoB | T
{'gene': 'katG', 'position': '944', 'ref': 'G', 'alt': 'C', 'quality': '38.7'}
944
katG | C
{'gene': 'gyrA', 'position': '248', 'ref': 'A', 'alt': 'G', 'quality': '52.1'}
248
gyrA | G
{'gene': 'gyrA', 'position': '259', 'ref': 'T', 'alt': 'C', 'quality': '29.4'}
259
gyrA | C
{'gene': 'pncA', 'position': '169', 'ref': 'G', 'alt': 'A', 'quality': '41.0'}
169
pncA | A


### 6D.2 (harder — convert and filter)
The `quality` column arrives as a **string**. Convert it to `float` and print only variants with quality **>= 40.0**, formatted as `gene position quality`.

In [58]:
# your code here
with open("data/variants.csv") as f:
    reader = csv.DictReader(f)
    for l in reader:
        q = float(l["quality"])
        if q >= 40:
            print(f"{l["gene"]} | {l["position"]} | {l["quality"]}")

rpoB | 1349 | 45.2
gyrA | 248 | 52.1
pncA | 169 | 41.0


### Module 6 CHALLENGE — FASTA → GC report writer
Read `data/sample.fasta`, compute GC content per gene, and **write** a tab-separated file `gc_report.tsv` with a header row `gene\tgc` and one row per gene (GC as a 3-decimal fraction). Then read your file back and print it to confirm. This ties reading + writing together.

In [87]:
# your code here
def write_gc_table(fa, table):
    fasta = read_fasta(fa)
    with open(table, "w") as f:
        f.write("gene\tgc\n")
        for g, s in fasta.items():
            gc = (s.count("G") + s.count("C")) / len(s)
            f.write(f"{g.split()[0]}\t{gc:.3f}\n")

write_gc_table("data/sample.fasta", "data/gc_report.tsv")

with open("data/gc_report.tsv") as f:
    for l in f:
        line = l.strip("\n")
        print(line)

gene	gc
rpoB	0.442
katG	0.733
gyrA	0.509


---
# Module 7 — Errors, debugging & testing

## 7A. Read tracebacks
### Worked example (this cell is *meant* to error — read the traceback bottom-up)

In [16]:
seq = "ATGCGT"
print(seq[10])   # IndexError

IndexError: string index out of range

### 7A.1 (easy)
Write code that deliberately triggers each of these, one per line in **separate** cells if you like, and note the error type from the traceback: (a) a `KeyError`, (b) a `ValueError` from `int("abc")`, (c) a `ZeroDivisionError`. You don't need to handle them yet — just observe.

In [ ]:
# (a) KeyError
d = {"A": 1}
# your code here


In [ ]:
# (b) ValueError
# your code here


In [ ]:
# (c) ZeroDivisionError
# your code here


## 7B. try/except & raise
### Worked example

In [ ]:
def safe_gc(seq):
    try:
        seq = seq.upper()
        return (seq.count("G") + seq.count("C")) / len(seq)
    except ZeroDivisionError:
        return None

print(safe_gc("ATGC"), safe_gc(""))

### 7B.1 (easy)
Write `safe_open(path)` that tries to open and read a file, returning its length, but catches `FileNotFoundError` and returns the string `"missing"` instead. Test on `"data/sample.fasta"` and on `"data/nope.txt"`.

In [ ]:
# your code here


### 7B.2 (harder — validate & raise)
Write `gc_content(seq)` that:
- raises `ValueError` if the sequence contains anything other than A/C/G/T (case-insensitive),
- raises `ValueError` if the sequence is empty,
- otherwise returns the GC fraction.

Test it: call it on `"ATGC"` (works), and use try/except to show the error message for `"ATBX"` and for `""`.

In [ ]:
# your code here


## 7C. Debugging
### Worked example — a planted bug
The function below is supposed to count codons but gives the wrong answer for `"ATGAAAC"`. Run it, then debug.

In [ ]:
def count_codons_buggy(seq):
    count = 0
    for i in range(0, len(seq), 3):
        count += 1
    return count

print(count_codons_buggy("ATGAAAC"))   # returns 3, but only 2 *complete* codons exist

### 7C.1 (easy — find & fix)
Fix `count_codons` so it counts only **complete** codons (groups of exactly 3). For `"ATGAAAC"` (7 bases) the answer should be 2. First add a `print(i, seq[i:i+3])` inside the loop to *see* the problem, then fix it.

In [ ]:
# your code here


### 7C.2 (harder — assertions)
Write `translate_frame(seq)` that asserts (with a message) the sequence length is a multiple of 3 before translating. Show it raising `AssertionError` on `"ATGAA"` and working on `"ATGAAA"` (you can return a placeholder like the codon list).

In [ ]:
# your code here


## 7D. Testing
### Worked example — assert-based checks

In [ ]:
def reverse_complement(seq):
    return seq.upper().translate(str.maketrans("ACGT", "TGCA"))[::-1]

assert reverse_complement("ATGC") == "GCAT"
assert reverse_complement("") == ""
print("checks passed")

### 7D.1 (easy)
Write at least four `assert` checks for a `gc_content` function: an all-GC sequence (1.0), an all-AT sequence (0.0), a mixed sequence, and a lowercase input. Define `gc_content` first (no validation needed here).

In [ ]:
# your code here


### 7D.2 (harder — pytest style)
Rewrite your checks as `pytest`-style test **functions** (named `test_*`) in this notebook and call them manually. Include one test using a try/except to confirm that an invalid input raises `ValueError` (simulating `pytest.raises`).

In real life these go in a `test_seqtools.py` file run with `pytest` from the terminal — but defining and calling them here proves the logic.

In [ ]:
# your code here


### Module 7 CHALLENGE — harden seqtools.py
Take your `seqtools.py` from Part I (or recreate the functions here) and:
1. add input validation to `gc_content` and `reverse_complement` (raise `ValueError` on invalid DNA),
2. write a set of `test_*` functions covering normal cases, edge cases (empty), and the error path,
3. run all your tests and print a summary.

This is the professional pattern you'll use for your capstone tool.

In [ ]:
# your code here


---
# Module 8 — Object-oriented programming

## 8A. Your first class
### Worked example

In [ ]:
class Sequence:
    def __init__(self, identifier, seq):
        self.id = identifier
        self.seq = seq.upper()
    def __len__(self):
        return len(self.seq)
    def gc_content(self):
        if len(self.seq) == 0:
            return None
        g = self.seq.count("G") + self.seq.count("C")
        return g / len(self.seq)

s = Sequence("rpoB", "atgcgc")
print(s.id, len(s), s.gc_content())

### 8A.1 (easy)
Extend the `Sequence` class with a `reverse_complement()` method that returns a **new** `Sequence` (id with `"_rc"` appended). Test: print the seq of `Sequence("g1","ATGC").reverse_complement()` (should be `GCAT`).

In [ ]:
# your code here — define the class, then test


### 8A.2 (harder)
Add a `transcribe()` method returning the mRNA string (T→U), and a `composition()` method returning a dict of base counts. Test both on `Sequence("g1", "ATGCGT")`.

In [ ]:
# your code here


## 8B. Dunder methods
### Worked example

In [ ]:
class Sequence:
    def __init__(self, identifier, seq):
        self.id, self.seq = identifier, seq.upper()
    def __len__(self): return len(self.seq)
    def __str__(self): return f"{self.id}: {self.seq}"
    def __repr__(self): return f"Sequence({self.id!r}, {self.seq!r})"
    def __getitem__(self, index): return self.seq[index]

s = Sequence("rpoB", "ATGCGC")
print(s)          # __str__
print(repr(s))    # __repr__
print(s[0:3])     # __getitem__

### 8B.1 (easy)
Build a `Sequence` class with `__init__`, `__len__`, and `__str__` so that `print(Sequence("g1","atgc"))` shows `g1: ATGC`.

In [ ]:
# your code here


### 8B.2 (harder)
Add `__eq__` so two `Sequence` objects are equal when their **sequences** match (ignore the id). Test: `Sequence("a","ATGC") == Sequence("b","atgc")` should be `True`, and `== Sequence("c","TTTT")` should be `False`.

In [ ]:
# your code here


## 8C. Inheritance
### Worked example

In [ ]:
class Sequence:
    def __init__(self, identifier, seq):
        self.id, self.seq = identifier, seq.upper()
    def gc_content(self):
        g = self.seq.count("G") + self.seq.count("C")
        return g / len(self.seq) if self.seq else None

class FastqRead(Sequence):
    def __init__(self, identifier, seq, qual):
        super().__init__(identifier, seq)
        self.qual = qual
    def mean_quality(self):
        scores = [ord(c) - 33 for c in self.qual]
        return sum(scores) / len(scores)

r = FastqRead("read1", "ATGC", "IIII")
print(r.gc_content(), r.mean_quality())

### 8C.1 (easy)
Create a `FastqRead` from the first read of `data/sample.fastq` (use your `read_fastq` from Module 6, or hardcode read1). Print its GC content (inherited) and mean quality (new).

In [ ]:
# your code here


### 8C.2 (harder)
Add a `passes_filter(self, min_q=30)` method to `FastqRead` returning `True` if mean quality >= `min_q`. Test on a high-quality and a low-quality read.

In [ ]:
# your code here


---
## PART II CAPSTONE — object-oriented FASTA toolkit
Build a small, tested, object-oriented module. In code:
1. A `Sequence` class with `__init__`, `__len__`, `__str__`, `gc_content()`, `reverse_complement()`, and input validation (raise `ValueError` on non-ACGT).
2. A function `load_fasta(path)` that parses a FASTA file and returns a **list of `Sequence` objects**.
3. At least three `test_*` functions: one for `gc_content`, one for `reverse_complement`, one confirming invalid input raises `ValueError`. Run them.
4. Use `load_fasta("data/sample.fasta")` and print a report: for each `Sequence`, its id, length, and GC%.

When this runs cleanly with passing tests, you've completed Part II — you can read real files, write robust validated code with tests, and structure it with classes. This is essentially the skeleton of a real bioinformatics tool.

In [ ]:
# your code here
